<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/3feat-encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## preprocessing/
## Tiền xử lý bước 3
## Bước này tạo files:
- text_feat.npy (xài sentence_transformer)
- image_feat.npy (đọc file image_features_Sports_and_Outdoors.b tìm có text mà không có hình fill giá trị mặc định)
- missed_img_itemIDs.csv (có text mà không có hình)

## ref:
- https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

# Sports14 Text/Image Feature Extraction

In [1]:
import os
import numpy as np
import pandas as pd


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec"

In [ ]:
!mkdir "{PATH}/data/3feat-encoder"

In [4]:
!cp "{PATH}/data/2reindex-feat/meta-sports14.csv" .

## Load text data

In [5]:
i_id, desc_str = 'itemID', 'description'

file_path = './'
filename = 'meta-sports14.csv'

meta_file = os.path.join(file_path, filename)

df = pd.read_csv(meta_file)
df.sort_values(by=[i_id], inplace=True)

print('data loaded!')
print(f'shape: {df.shape}')

df[:3]

data loaded!
shape: (18357, 10)


,itemID,asin,title,price,imUrl,related,brand,categories,salesRank,description
0,0,1881509818,Ghost Inc Glock Armorers Tool 3/32 Punch,9.99,http://ecx.images-amazon.com/images/I/21iMxsyD...,"{'also_bought': ['B000U3YWEM', 'B000U401J6', '...",Ghost,"[['Sports & Outdoors', 'Hunting & Fishing', 'H...",{'Sports &amp; Outdoors': 172909},Ghost Armorer Tool (1). The GAT is made with a...
1,1,2094869245,5 LED Bicycle Rear Tail Red Bike Torch Laser B...,8.26,http://ecx.images-amazon.com/images/I/51RtwnJw...,"{'also_bought': ['B0081O93N2', 'B00EYTCHJA', '...",NaN,"[['Sports & Outdoors', 'Cycling', 'Lights & Re...",{'Sports &amp; Outdoors': 14293},This newly-designed Laser tail light can emit ...
2,2,7245456259,Black Mountain Products Single Resistance Band...,10.49,http://ecx.images-amazon.com/images/I/411Ikpf1...,"{'also_bought': ['B00DDBS2JE', 'B00H1KNHE8', '...",Black Mountain,"[['Sports & Outdoors', 'Exercise & Fitness', '...",{'Sports &amp; Outdoors': 1010},Black Mountain Products single resistance band...


Trước khi bạn dùng các mô hình ngôn ngữ (như BERT hay Sentence-Transformer) để tạo vector đặc trưng, bạn cần biết có bao nhiêu sản phẩm bị thiếu thông tin. Nếu một sản phẩm không có tiêu đề, không có mô tả, cũng không có thương hiệu, thì mô hình sẽ không có "nguyên liệu" để tạo ra vector, dẫn đến việc gợi ý cho sản phẩm đó sẽ rất kém.

Code đang kiểm tra sự thiếu hụt (null) của 4 trường thông tin quan trọng: title (tiêu đề), description (mô tả), brand (thương hiệu), và categories (danh mục).

In [ ]:
idx = 6
print(df.iloc[idx]["title"])
print(df.iloc[idx]["imUrl"])

In [6]:
# sentences: title + brand + category + description | All have title + description

title_na_df = df[df['title'].isnull()]
print(title_na_df.shape)

desc_na_df = df[df['description'].isnull()]
print(desc_na_df.shape)

na_df = df[df['description'].isnull() & df['title'].isnull()]
print(na_df.shape)

na3_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull()]
print(na3_df.shape)

na4_df = df[df['description'].isnull() & df['title'].isnull() & df['brand'].isnull() & df['categories'].isnull()]
print(na4_df.shape)

(91, 10)
(2659, 10)
(40, 10)
(40, 10)
(0, 10)


Đoạn code này là bước "Cứu vãn dữ liệu rỗng" (Data Cleaning) ngay sau bước kiểm tra mà bạn vừa thực hiện ở trên.

Trong lập trình, đặc biệt là khi làm việc với văn bản (NLP), giá trị NaN (giá trị rỗng/null) giống như một "viên đá ngáng đường". Nếu bạn đưa một giá trị NaN vào mô hình BERT hay bất kỳ hàm xử lý chuỗi nào, chương trình sẽ bị báo lỗi (Crash) ngay lập tức vì nó không biết cách xử lý một thực thể "không tồn tại".

In [7]:
df[desc_str] = df[desc_str].fillna(" ")
df['title'] = df['title'].fillna(" ")
df['brand'] = df['brand'].fillna(" ")
df['categories'] = df['categories'].fillna(" ")


Đoạn code này chính là bước "đúc kết" tất cả các trường thông tin rời rạc thành một câu văn hoàn chỉnh cho mỗi sản phẩm. Mục tiêu là tạo ra một danh sách các chuỗi văn bản (sentences) để chuẩn bị nạp vào mô hình AI (như BERT) nhằm trích xuất vector đặc trưng.

In [8]:
sentences = []
for i, row in df.iterrows():
    sen = row['title'] + ' ' + row['brand'] + ' '

    # eval(row['categories']): Trong file Metadata của Amazon, cột danh mục thường được lưu dưới dạng chuỗi trông giống list
    # (ví dụ: "[['Sports', 'Tennis', 'Rackets']]"). Hàm eval sẽ chuyển chuỗi này thành một List thực thụ trong Python.
    cates = eval(row['categories'])
    if isinstance(cates, list):
        for c in cates[0]:
            sen = sen + c + ' '
    sen += row[desc_str]
    sen = sen.replace('\n', ' ')

    sentences.append(sen)

sentences[:5]

['Ghost Inc Glock Armorers Tool 3/32 Punch Ghost Sports & Outdoors Hunting & Fishing Hunting Gun Maintenance Gunsmithing Tools Ghost Armorer Tool (1). The GAT is made with a spring steel punch. The diameter is 3/32 of an inch or 2.5mm, this is the same as the OEM tool size. The difference is you will be able to press harder without bending the shaft of this punch. Just a better tool to work on your Glock with.',
 '5 LED Bicycle Rear Tail Red Bike Torch Laser Beam Lamp Light   Sports & Outdoors Cycling Lights & Reflectors Taillights This newly-designed Laser tail light can emit two parallel lines, to form a virtual lane together with the moving of bicycle on the road. LED flash light and  two lines not only enhance the waring effect strongly and greatly but also improve the safety of night riding.',
 'Black Mountain Products Single Resistance Band - Door Anchor and Starter Guide Included Black Mountain Sports & Outdoors Exercise & Fitness Accessories Exercise Bands Black Mountain Produc

In [9]:
course_list = df[i_id].tolist()
#sentences = df[desc_str].tolist()

assert course_list[-1] == len(course_list) - 1

In [10]:
# Dữ liệu này có mấy tag html hay word
sentences[3:4]

['Black Mountain Products Resistance Band Set with Door Anchor, Ankle Strap, Exercise Chart, and Resistance Band Carrying Case Black Mountain Sports & Outdoors Exercise & Fitness Accessories Exercise Bands [if gte mso 9]><xml> <o:OfficeDocumentSettings> <o:AllowPNG  /> </o:OfficeDocumentSettings> </xml><![endif][if gte mso 9]><xml> <w:WordDocument> <w:View>Normal</w:View> <w:Zoom>0</w:Zoom> <w:TrackMoves  /> <w:TrackFormatting  /> <w:PunctuationKerning  /> <w:ValidateAgainstSchemas  /> <w:SaveIfXMLInvalid>false</w:SaveIfXMLInvalid> <w:IgnoreMixedContent>false</w:IgnoreMixedContent> <w:AlwaysShowPlaceholderText>false</w:AlwaysShowPlaceholderText> <w:DoNotPromoteQF  /> <w:LidThemeOther>EN-US</w:LidThemeOther> <w:LidThemeAsian>X-NONE</w:LidThemeAsian> <w:LidThemeComplexScript>X-NONE</w:LidThemeComplexScript> <w:Compatibility> <w:BreakWrappedTables  /> <w:SnapToGridInCell  /> <w:WrapTextWithPunct  /> <w:UseAsianBreakRules  /> <w:DontGrowAutofit  /> <w:SplitPgBreakAndParaMark  /> <w:EnableO

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Tính độ dài (số từ) của mỗi câu
doc_lengths = [len(s.split()) for s in sentences]

# 2. Vẽ biểu đồ
plt.figure(figsize=(10, 5))
plt.hist(doc_lengths, bins=50, color='skyblue', edgecolor='black')
plt.title('Phân bổ độ dài văn bản sản phẩm')
plt.xlabel('Số lượng từ')
plt.ylabel('Số lượng sản phẩm')
plt.axvline(x=256, color='red', linestyle='--', label='Giới hạn BERT (256)')
plt.legend()
plt.show()

# 3. Thống kê nhanh
print(f"Độ dài trung bình: {sum(doc_lengths)/len(doc_lengths):.2f}")
print(f"Số sản phẩm dài hơn 256 từ: {sum(1 for l in doc_lengths if l > 256)}")

In [ ]:
from collections import Counter
import re

# Gộp tất cả thành một chuỗi khổng lồ, chuyển về chữ thường
all_text = " ".join(sentences).lower()
# Chỉ lấy các từ (loại bỏ dấu câu)
words = re.findall(r'\w+', all_text)

# Đếm tần suất
word_counts = Counter(words)
print("Top 20 từ phổ biến nhất:")
for word, count in word_counts.most_common(20):
    print(f"{word}: {count}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import seaborn as sns
import matplotlib.pyplot as plt

def plot_top_ngrams(corpus, n=2, top_k=20):
    # Khởi tạo vectorizer để đếm cụm n-từ
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)

    # Lấy tần suất
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)[:top_k]

    # Vẽ biểu đồ
    df_ngram = pd.DataFrame(words_freq, columns=['Text', 'Count'])
    plt.figure(figsize=(10,8))
    sns.barplot(x='Count', y='Text', data=df_ngram, palette='viridis')
    plt.title(f'Top {top_k} Bigrams trong dữ liệu Sports')
    plt.show()

# Chạy thử với danh sách sentences của bạn
plot_top_ngrams(sentences, n=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Tính toán số lượng từ cho mỗi sản phẩm
doc_lengths = [len(s.split()) for s in sentences]

# 2. Khởi tạo khung hình
plt.figure(figsize=(12, 6))

# 3. Vẽ Box Plot nằm ngang
sns.boxplot(x=doc_lengths, color='lightgreen', flierprops={"marker": "x", "markerfacecolor": "red"})

# 4. Thêm các đường thông số quan trọng
mean_val = np.mean(doc_lengths)
plt.axvline(mean_val, color='blue', linestyle='--', label=f'Trung bình: {mean_val:.2f}')

# 5. Tinh chỉnh tiêu đề và nhãn
plt.title('Biểu đồ hộp (Box Plot) về độ dài văn bản sản phẩm', fontsize=15)
plt.xlabel('Số lượng từ', fontsize=12)
plt.legend()

# 6. Hiển thị các chỉ số thống kê cơ bản lên màn hình
quartiles = np.percentile(doc_lengths, [25, 50, 75])
print(f"Giá trị nhỏ nhất: {min(doc_lengths)}")
print(f"Tứ phân vị Q1 (25%): {quartiles[0]}")
print(f"Trung vị (50%): {quartiles[1]}")
print(f"Tứ phân vị Q3 (75%): {quartiles[2]}")
print(f"Giá trị lớn nhất: {max(doc_lengths)}")

plt.show()

In [ ]:
!pip install sentence_transformers

In [ ]:
# should `pip install sentence_transformers` first
# should use gpu to speed up
# https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

from sentence_transformers import SentenceTransformer
# all-MiniLM-L6-v2, all-MiniLM-L12-v2, all-mpnet-base-v2 gpu (31s) (36s) (3p)

model_name = "all-MiniLM-L12-v2"

model = SentenceTransformer(model_name)

In [3]:
# !pip install fastembed -q

In [4]:
# from fastembed import TextEmbedding

# # https://qdrant.github.io/fastembed/examples/Supported_Models/#supported-text-embedding-models
# model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
# print("Đang encode...")

# embeddings_generator = model.embed(sentences)
# sentence_embeddings = np.array(list(embeddings_generator))

# print(f"Shape: {sentence_embeddings.shape}")


# np.save(os.path.join('text_feat.npy'), sentence_embeddings)
# print('Đã lưu xong bằng FastEmbed!')

In [ ]:
# import re
# import numpy as np

# # 1. Hàm làm sạch nhanh
# def clean_text(text):
#     text = re.sub(r'<.*?>', '', text)  # Xóa thẻ <...>
#     text = re.sub(r'\[.*?\]', '', text) # Xóa mã [if ...]
#     text = text.replace('\n', ' ')      # Xóa xuống dòng
#     return ' '.join(text.split())       # Xóa khoảng trắng thừa

# # 2. Làm sạch toàn bộ danh sách sentences
# print("Cleaning sentences...")
# cleaned_sentences = [clean_text(s) for s in sentences]

# # 3. Tiến hành encode (Dùng danh sách đã sạch)
# print("Encoding text...")
# sentence_embeddings = model.encode(cleaned_sentences, show_progress_bar=True)

# print('text encoded!')
# assert sentence_embeddings.shape[0] == df.shape[0]

# # 4. Lưu lại
# np.save(os.path.join(file_path, 'text_feat.npy'), sentence_embeddings)
# print('done!')

In [ ]:
!nvidia-smi

In [ ]:
import torch
import os
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Kiểm tra GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Đang sử dụng thiết bị: {device.upper()}")

# 2. Load model vào thiết bị đã chọn
model = SentenceTransformer(model_name, device=device)

# 3. Tiến hành encode (SBERT tự động xử lý hàng loạt cực nhanh trên GPU)
print("Bắt đầu Encode... Vui lòng đợi trong giây lát.")
sentence_embeddings = model.encode(sentences, show_progress_bar=True, batch_size=64)

print(f'Text encoded! Shape: {sentence_embeddings.shape}')

# 4. Lưu kết quả
assert sentence_embeddings.shape[0] == df.shape[0]
save_path = os.path.join(file_path, 'text_feat.npy')
np.save(save_path, sentence_embeddings)

print(f'Done! File đã được lưu tại: {save_path}')

In [ ]:
!pip install nvitop

In [ ]:
!nvitop

In [ ]:
# sentence_embeddings = model.encode(sentences)
# print('text encoded!')

# assert sentence_embeddings.shape[0] == df.shape[0]
# np.save(os.path.join(file_path, 'text_feat.npy'), sentence_embeddings)
# print('done!')

In [ ]:
!cp "text_feat.npy" "{PATH}/data/3feat-encoder/text_feat.{model_name}.npy"

In [ ]:
sentence_embeddings[:10]

In [ ]:
!cp "{PATH}/text_feat.all-MiniLM-L6-v2.npy" "text_feat.npy"

In [ ]:
load_txt_feat = np.load('text_feat.npy', allow_pickle=True)
print(load_txt_feat.shape)
load_txt_feat[:10]

# Image encoder (V0)，following LATTICE, averaging over for missed items

In [ ]:
df[:5]

In [ ]:
!wget "https://snap.stanford.edu/data/amazon/productGraph/image_features/categoryFiles/image_features_Sports_and_Outdoors.b"

In [ ]:
!cat "{PATH}/data/3feat-encoder/"

In [ ]:
!cp "image_features_Sports_and_Outdoors.b" "{PATH}/data/3feat-encoder/image_features_Sports_and_Outdoors.b"

In [ ]:
!cp "{PATH}/data/3feat-encoder/image_features_Sports_and_Outdoors.b" "image_features_Sports_and_Outdoors.b"

In [ ]:
import array

# https://cseweb.ucsd.edu/~jmcauley/datasets/amazon/links.html
def readImageFeatures(path):
  f = open(path, 'rb')
  while True:
    asin = f.read(10).decode('UTF-8')
    if asin == '': break
    a = array.array('f')
    a.fromfile(f, 4096)
    yield asin, a.tolist()

In [ ]:
img_data = readImageFeatures("image_features_Sports_and_Outdoors.b")

In [ ]:
img_data

In [ ]:
# Lấy thử 1 phần tử đầu tiên
first_item = next(img_data)

# Kiểm tra hình dạng
print(f"Kiểu dữ liệu của 1 phần tử: {type(first_item)}")
print(f"Mã ASIN (ID sản phẩm): {first_item[0]}")
print(f"Độ dài vector ảnh: {len(first_item[1])}")
print(f"5 giá trị đầu tiên trong vector: {first_item[1][:5]}")

In [ ]:
first_item[1][:10]

In [ ]:
# 1. Khởi tạo lại generator để đọc từ đầu file
img_data = readImageFeatures("image_features_Sports_and_Outdoors.b")

filename = "image_features_Sports_and_Outdoors.sample.b"

# 2. Mở file để ghi
with open(filename, "w", encoding="utf-8") as f:
    f.write("HÌNH DẠNG DỮ LIỆU 5 DÒNG ĐẦU TIÊN CỦA IMG_DATA\n")
    f.write("="*50 + "\n\n")

    # 3. Lấy 5 dòng đầu tiên
    for i in range(5):
        try:
            asin, feature_vector = next(img_data)

            f.write(f"Dòng {i+1}:\n")
            f.write(f"- Mã ASIN (ID): {asin}\n")
            f.write(f"- Độ dài vector: {len(feature_vector)}\n")
            # Ghi thử 10 giá trị đầu của vector để xem dạng số
            f.write(f"- 10 giá trị đầu của vector: {feature_vector[:10]}\n")
            f.write("-" * 30 + "\n")
        except StopIteration:
            f.write("Hết dữ liệu trong file!\n")
            break

print(f"Đã ghi 5 dòng đầu tiên vào file {filename} thành công!")

In [ ]:
# Mở file gốc để đọc
f_in = open("image_features_Sports_and_Outdoors.b", "rb")
# Mở file mới để ghi (chế độ binary)
filename = "image_features_Sports_and_Outdoors.sample.b"
f_out = open(filename, "wb")

# Mỗi dòng gồm 10 byte (ASIN) + 4096 số float (mỗi số 4 byte)
# Tổng cộng byte mỗi dòng = 10 + (4096 * 4) = 16394 byte
bytes_per_row = 10 + (4096 * 4)

# Đọc và ghi đúng 5 dòng
for _ in range(5):
    data = f_in.read(bytes_per_row)
    if not data:
        break
    f_out.write(data)

f_in.close()
f_out.close()
print(f"Đã cắt 5 dòng đầu tiên và lưu vào file {filename}")

In [ ]:
import json

# Khởi tạo lại để đọc từ đầu
img_data = readImageFeatures("image_features_Sports_and_Outdoors.b")

first_5_list = []
for i in range(5):
    try:
        first_5_list.append(next(img_data))
    except StopIteration:
        break

# Ghi ra file dạng JSON để giữ đúng cấu trúc list
with open("image_features_Sports_and_Outdoors.sample.json", "w") as f:
    json.dump(first_5_list, f)

print("Đã lưu 5 dòng dữ liệu (ASIN + List) vào file 'image_features_Sports_and_Outdoors.sample.json'")

In [ ]:
!cp "image_features_Sports_and_Outdoors.sample.b" "{PATH}/data/3feat-encoder/image_features_Sports_and_Outdoors.sample.b"

In [ ]:
!cp "image_features_Sports_and_Outdoors.sample.json" "{PATH}/data/3feat-encoder/image_features_Sports_and_Outdoors.sample.json"

In [ ]:
# 1. Giai đoạn 1: Lọc và Thu thập dữ liệu hiện có
item2id = dict(zip(df['asin'], df['itemID']))

feats = {}
avg = []
for d in img_data:
    if d[0] in item2id:
        feats[int(item2id[d[0]])] = d[1]
        avg.append(d[1])

Vòng lặp for d in img_data: Duyệt qua từng sản phẩm lấy ra từ file nhị phân .b.

d[0] là mã ASIN, d[1] là vector đặc trưng (4096 chiều).

if d[0] in item2id: Chỉ lấy những sản phẩm nào có nằm trong file metadata của bạn (vì file ảnh có thể chứa nhiều sản phẩm không liên quan).

feats[int(item2id[d[0]])] = d[1]: Lưu vector vào từ điển với "chìa khóa" là itemID (số thứ tự 0, 1, 2...). Điều này giúp truy xuất cực nhanh.

avg.append(d[1]): Gom tất cả các vector tìm thấy vào một danh sách để chuẩn bị tính trung bình.

In [ ]:
# 2. Giai đoạn 2: Tính toán "Vector trung bình" (Imputation)
avg = np.array(avg).mean(0).tolist()

Tại sao phải làm vậy?: Trong thực tế, có những sản phẩm bạn có ID nhưng trong file ảnh lại bị thiếu. Để mô hình GCN không bị lỗi, bạn không thể để trống.

Giải pháp: Tính giá trị trung bình của tất cả các ảnh hiện có (mean(0)). Vector này đại diện cho "đặc trưng ảnh chung chung nhất".

In [ ]:
# 3. Giai đoạn 3: Lấp đầy khoảng trống (Final Mapping)
ret = []
non_no = []
for i in range(len(item2id)):
    if i in feats:
        ret.append(feats[i]) # Nếu có ảnh: lấy ảnh thật
    else:
        non_no.append(i) # Nếu thiếu ảnh: lưu lại ID bị thiếu
        ret.append(avg) # ...và gán cho nó vector trung bình

print('# of items not in processed image features:', len(non_no))
assert len(ret) == len(item2id)
np.save('image_feat.npy', np.array(ret))
np.savetxt("missed_img_itemIDs.csv", non_no, delimiter =",", fmt ='%d') # save for report missing data
print('done!')

In [ ]:
!cp "image_feat.npy" "{PATH}/data/3feat-encoder/"
!cp "missed_img_itemIDs.csv" "{PATH}/data/3feat-encoder/"

In [ ]:
!pip freeze > requirements.txt